Q1. List all bookings made by a person named Darren Smith as the following.
```
member_id | first_name | last_name | address | facility_id | slots
---------------------------------------------------------------------
```

Ensure the following
1. Show the details of all persons named Darren Smith even if they have not made any bookings
2. Sort the result by number of slots (highers first)
3. List the person with no bookings at the top

In [0]:
from pyspark.sql.functions import expr, col
member_df = spark.table("dev_catalog.spark_db.members").alias("m")
bookings_df = spark.table("dev_catalog.spark_db.bookings").alias("b")

join_condition_expr = expr("m.member_id = b.member_id")
outer_join_df = ( member_df
                 .join(bookings_df,join_condition_expr,'left')
                #  Use filter or where clause
                 .filter("m.first_name =='Darren' and m.last_name =='Smith'")
                 .selectExpr("m.member_id","m.first_name", "m.last_name", "m.address", "b.facility_id", "b.slots")
                #  .select("m.member_id", "m.first_name", "m.last_name","m.address" ,"b.facility_id","b.slots")
                 .orderBy(col("slots").desc_nulls_first())

)
outer_join_df.display()

Q2. Show me a bookings report for Darren Smith as the following.

```
facility_name | slots | booking_amount | start_time | member_id | member_name | telephone | address
------------------------------------------------------------------------------------------------------
```
The report must meet the following criteria.

1. Show the details of all persons named Darren Smith even if they have not made any bookings
2. Sort the result by number of slots (highers first)
3. List the person with no bookings at the top

In [0]:

members_df = (
    spark.table("dev_catalog.spark_db.members")
    .where("first_name == 'Darren' and last_name == 'Smith'")
    
)

bookings_df = (
    spark.table("dev_catalog.spark_db.bookings").alias("b")
)
facilities_df = (
    spark.table("dev_catalog.spark_db.facilities").alias('f')
)

joined_df = (
     members_df.alias('m')
     .join(bookings_df, expr("m.member_id = b.member_id"),'left')
     .join(facilities_df.alias('f'),expr("b.facility_id = f.facility_id"),'left')
 )

result_df = (
    joined_df
    .select(
        "f.facility_name",
        expr("b.slots * f.member_cost as booking_amount"),
        "b.start_time", "b.member_id",
        expr("concat_ws(' ', m.first_name, m.last_name)").alias("member_name"),
        "m.telephone", "m.address"
    )
    .orderBy(col("slots").desc_nulls_first())
)

result_df.display()


Q3. Prepare a facility booking report as the following
```
facility_name | member_cost | gest_cost | start_time | slots
---------------------------------------------------------------
```
Ensure the following
1. All club facilities must be listed in the report
2. Consider only bookings for more than 10 slots

In [0]:
from pyspark.sql.functions import expr
facilities_df = spark.table("dev_catalog.spark_db.facilities") 
# bookings_df = spark.table("dev_catalog.spark_db.bookings").filter("slots > 10")
bookings_df = spark.table("dev_catalog.spark_db.bookings")

result_df = (
    facilities_df.alias('f')
    .join(bookings_df.alias('b'),expr("b.facility_id = f.facility_id and b.slots> 10"),'left')
    .select(
        "f.facility_name",
        "f.member_cost",
        "f.guest_cost",
        "b.start_time",
        "b.slots"
    )
)

result_df.display()


Q4. Prepare a member bookings report as the following
```
booking_id | facility_name | slots | first_name | last_name | address
```
Ensure the following
1. Consider only regular memebrs (not guest) and direct members(not recomended by any other member)
2. Consider only bookings for more than 8 hours
3. Ensure all regular and direct members are listed even if they have no 8 hour bookings
4. Ensure all 8 hour bookings are listed even if they are not made by regular and direct members
5. Sort the report by slots and first name in ascending order

In [0]:
from pyspark.sql.functions import expr

members_df = (
    spark.table("dev_catalog.spark_db.members")
        .filter("member_id != 0 and recommended_by is null")
        .alias("m")
)

bookings_df = (
    spark.table("dev_catalog.spark_db.bookings")
        .filter("slots > 8")
        .alias("b")
)

facilities_df = spark.table("dev_catalog.spark_db.facilities").alias("f")

full_join_df = members_df.join(bookings_df, expr("m.member_id == b.member_id"), "full")

result_df = (
    full_join_df.join(facilities_df, expr("b.facility_id == f.facility_id"), "left")
    .select("b.booking_id","f.facility_name","b.slots","m.first_name","m.last_name","m.address")
    .orderBy(expr("b.slots").asc_nulls_last(), expr("m.first_name").asc_nulls_last())
)

display(result_df)